Audyt metodologii 2026-09-10: wyniki historyczne unieważnione. Definicje i ograniczenia: `../docs/methodology_audit.md`. Przeliczenia lokalne: `../data/processed/audit_v2/`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

df_path = DATA_PROCESSED / "df_model_clean_v1.parquet"
rapping_path = DATA_PROCESSED / "rapping_features_v1.parquet"

df = pd.read_parquet(df_path)
# Recomputed below: rapping

print("df:", df.shape)

print(df.index.min(), "→", df.index.max())
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.time_analysis import (validate_time, time_shift, past_mean, lagged_corr,
    rapping_starts, rapping_features, complete_window, tail_mask as post_tail_mask, coverage)

validate_time(df)
rapping = rapping_features(df["008B05154"])


In [ ]:
TARGET_COL = "008A01345"  # dust concentration after ESP

U_COLS = {
    "U1": "008A02289",
    "U2": "008A02290",
    "U3": "008A02291",
}

I_COLS = {
    "I1": "008A02267",
    "I2": "008A02268",
    "I3": "008A02269",
}

P_COLS = {
    "P1": "008A02273",
    "P2": "008A02274",
    "P3": "008A02275",
}

RAPPING_TIME_COL = "minutes_since_rapping_3_collecting_start_0_5"
RAPPING_FLAG_COL = "is_within_5min_after_rapping_3_collecting"

required_cols = (
    [TARGET_COL]
    + list(U_COLS.values())
    + list(I_COLS.values())
    + list(P_COLS.values())
)

missing = [col for col in required_cols if col not in df.columns]
missing

In [ ]:
df_ov = df.copy()

# Dołączamy cechy strzepywania, jeżeli nie ma ich jeszcze w df
rapping_cols = [RAPPING_TIME_COL, RAPPING_FLAG_COL]

df_ov = df_ov.join(
    rapping[rapping_cols],
    how="left"
)

# Suma mocy i średnie napięcie
df_ov["P_total"] = df_ov[list(P_COLS.values())].sum(axis=1)
df_ov["U_mean"] = df_ov[list(U_COLS.values())].mean(axis=1)
df_ov["U_sum"] = df_ov[list(U_COLS.values())].sum(axis=1)

# Przybliżona moc z U*I dla kontroli
for sec in ["1", "2", "3"]:
    df_ov[f"P{sec}_calc"] = (
        df_ov[U_COLS[f"U{sec}"]] * df_ov[I_COLS[f"I{sec}"]] / 1000
    )

df_ov["P_total_calc"] = df_ov[["P1_calc", "P2_calc", "P3_calc"]].sum(axis=1)

df_ov[
    [TARGET_COL, "P_total", "P_total_calc", "U_mean", "U_sum"]
].describe()

In [ ]:
voltage_summary = df_ov[
    list(U_COLS.values()) + ["U_mean", "U_sum"]
].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

voltage_summary

In [ ]:
for name, col in U_COLS.items():
    plt.figure(figsize=(8, 4))
    plt.hist(df_ov[col].dropna(), bins=60)
    plt.xlabel(f"{name} voltage [kV]")
    plt.ylabel("Count")
    plt.title(f"Distribution of {name} voltage")
    plt.grid(True)
    plt.show()

In [ ]:
plt.figure(figsize=(14, 5))

for name, col in U_COLS.items():
    plt.plot(df_ov.index, df_ov[col], label=name, linewidth=0.8)

plt.xlabel("Time")
plt.ylabel("Voltage [kV]")
plt.title("ESP section voltages over time")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
samples_per_min = 6

for window_min in [1, 5, 15]:
    shift_n = window_min * samples_per_min
    
    for name, col in U_COLS.items():
        df_ov[f"{name}_delta_{window_min}min"] = df_ov[col] - time_shift(df_ov[col], shift_n)
    
    df_ov[f"U_mean_delta_{window_min}min"] = (
        df_ov["U_mean"] - time_shift(df_ov["U_mean"], shift_n)
    )

delta_cols = [
    col for col in df_ov.columns
    if "delta" in col and ("U" in col)
]

df_ov[delta_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

In [ ]:
voltage_change_rows = []

for window_min in [1, 5, 15]:
    for name in ["U1", "U2", "U3", "U_mean"]:
        delta_col = f"{name}_delta_{window_min}min"
        
        for threshold in [0.5, 1.0, 2.0, 5.0]:
            frac = (df_ov[delta_col].dropna().abs() >= threshold).mean()
            
            voltage_change_rows.append({
                "signal": name,
                "window_min": window_min,
                "threshold_kv": threshold,
                "fraction_abs_delta_above_threshold": frac,
            })

voltage_change_summary = pd.DataFrame(voltage_change_rows)

voltage_change_summary.pivot_table(
    index=["signal", "window_min"],
    columns="threshold_kv",
    values="fraction_abs_delta_above_threshold"
)

In [ ]:
non_rapping_mask = (
    df_ov[RAPPING_FLAG_COL].eq(0)
)

df_nonrap = df_ov.loc[non_rapping_mask].copy()

print("All samples:", len(df_ov))
print("Non-rapping samples:", len(df_nonrap))
print("Fraction non-rapping:", len(df_nonrap) / len(df_ov))

In [ ]:
# Pomocniczo: pokaż kolumny, które mogą być procesowe
candidate_process_keywords = [
    "016A",  # kocioł / proces
    "008A007",  # część temperatur / O2 / inne ESP, zależnie od mappingu
    "008A009",
]

candidate_cols = [
    col for col in df_nonrap.columns
    if any(key in col for key in candidate_process_keywords)
]

candidate_cols[:100], len(candidate_cols)

In [ ]:
exclude_cols = set(
    [TARGET_COL, "P_total", "P_total_calc", "U_mean", "U_sum"]
    + list(U_COLS.values())
    + list(I_COLS.values())
    + list(P_COLS.values())
    + [RAPPING_TIME_COL, RAPPING_FLAG_COL]
)

# Wykluczamy też kolumny wyliczone w tym notebooku
exclude_patterns = ["delta", "_calc", "_error"]

numeric_cols = df_nonrap.select_dtypes(include=[np.number]).columns.tolist()

process_feature_candidates = []

for col in numeric_cols:
    if col in exclude_cols:
        continue
    if any(pattern in col for pattern in exclude_patterns):
        continue
    
    # odrzucamy kolumny prawie stałe
    if df_nonrap[col].nunique(dropna=True) < 10:
        continue
    
    process_feature_candidates.append(col)

print("Number of process feature candidates:", len(process_feature_candidates))
process_feature_candidates[:50]

In [ ]:
df_ov[[RAPPING_TIME_COL, RAPPING_FLAG_COL]].describe()

In [ ]:
df_ov[RAPPING_FLAG_COL].value_counts(dropna=False)

In [ ]:
# Poprawiona maska: poza znanym oknem 0–5 min po strzepywaniu
# Zakładamy, że RAPPING_FLAG_COL == 1 oznacza próbkę w oknie po strzepywaniu

non_rapping_mask = (
    df_ov[RAPPING_FLAG_COL].eq(0)
)

df_nonrap = df_ov.loc[non_rapping_mask].copy()

print("All samples:", len(df_ov))
print("Non-rapping samples:", len(df_nonrap))
print("Fraction non-rapping:", len(df_nonrap) / len(df_ov))

df_nonrap[[TARGET_COL, "P_total", "U_mean"]].describe()

In [ ]:
PROCESS_COLS_CANDIDATE = [
    "008A00936",
    "008A00910",
    "008A00911",
    "008A00720",
    "008A00728",
    "008A00719",
    "008A00727",
    "016A00219",
    "008A00723",
    "016A00396",
    "008A00731",
    "008A00741",
    "008A00742",
]

PROCESS_COLS_CANDIDATE = [
    col for col in PROCESS_COLS_CANDIDATE
    if col in df_nonrap.columns
]

process_summary = df_nonrap[PROCESS_COLS_CANDIDATE].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

process_summary["missing_frac"] = df_nonrap[PROCESS_COLS_CANDIDATE].isna().mean()

process_summary

In [ ]:
mapping_paths = [
    DATA_PROCESSED / "tag_mapping.csv",
    PROJECT_ROOT / "data" / "raw" / "tag_mapping.csv",
    PROJECT_ROOT / "tag_mapping.csv",
]

tag_mapping_path = None

for p in mapping_paths:
    if p.exists():
        tag_mapping_path = p
        break

tag_mapping_path

In [ ]:
tag_mapping = pd.read_csv(tag_mapping_path)

tag_mapping.head()

In [ ]:
tag_mapping.columns

In [ ]:
for col in tag_mapping.columns:
    print(col)

In [ ]:
PROCESS_COLS = [
    "016A00219",
    "016A00396",
    "008A00719",
    "008A00720",
    "008A00723",
    "008A00727",
    "008A00728",
    "008A00731",
    "008A00910",
    "008A00911",
]

PROCESS_COLS = [
    col for col in PROCESS_COLS
    if col in df_nonrap.columns
]

PROCESS_COLS

In [ ]:
df_nonrap[PROCESS_COLS].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

In [ ]:
overlap_cols = PROCESS_COLS + list(U_COLS.values()) + [TARGET_COL, "P_total", "U_mean"]

df_overlap = df_nonrap[overlap_cols].dropna().copy()

print("df_nonrap:", df_nonrap.shape)
print("df_overlap:", df_overlap.shape)
print("Fraction retained:", len(df_overlap) / len(df_nonrap))

In [ ]:
boiler_like_cols = [col for col in ["016A00219", "016A00396"] if col in df_overlap.columns]
boiler_like_cols

In [ ]:
for boiler_col in boiler_like_cols:
    tmp = df_overlap.copy()
    
    tmp["boiler_bin"] = pd.qcut(
        tmp[boiler_col],
        q=10,
        duplicates="drop"
    )
    
    grouped = tmp.groupby("boiler_bin").agg(
        n=("U_mean", "size"),
        U_mean_median=("U_mean", "median"),
        U_mean_std=("U_mean", "std"),
        U_mean_p05=("U_mean", lambda x: x.quantile(0.05)),
        U_mean_p95=("U_mean", lambda x: x.quantile(0.95)),
        dust_median=(TARGET_COL, "median"),
        P_total_median=("P_total", "median"),
    )
    
    grouped["U_mean_p95_minus_p05"] = grouped["U_mean_p95"] - grouped["U_mean_p05"]
    
    print("\nBoiler-like column:", boiler_col)
    display(grouped)

In [ ]:
for boiler_col in boiler_like_cols:
    plt.figure(figsize=(8, 5))
    
    plt.scatter(
        df_overlap[boiler_col],
        df_overlap["U_mean"],
        s=3,
        alpha=0.15
    )
    
    plt.xlabel(boiler_col)
    plt.ylabel("U_mean [kV]")
    plt.title(f"U_mean vs {boiler_col}")
    plt.grid(True)
    plt.show()

In [ ]:
tag_mapping = pd.read_csv(tag_mapping_path, sep=";")

tag_mapping.head()

In [ ]:
tag_mapping.columns

In [ ]:
process_tag_descriptions = (
    tag_mapping
    .loc[tag_mapping["Nazwa"].isin(PROCESS_COLS)]
    .copy()
)

process_tag_descriptions

In [ ]:
process_tag_descriptions = (
    tag_mapping
    .loc[tag_mapping["Nazwa"].isin(PROCESS_COLS)]
    .set_index("Nazwa")
    .loc[PROCESS_COLS]
    .reset_index()
)

process_tag_descriptions

### Status interpretacji overlap
Zmienność napięcia w binach obciążenia jest opisem danych obserwacyjnych. Nie dowodzi wymienialności punktów pracy ani bezpieczeństwa zmiany napięcia. Hipoteza lokalnego overlap wymaga sprawdzenia pozostałych zmiennych procesowych, trybu ECO i ograniczeń instalacji. Historyczne wartości liczbowe wymagają ponownego przeliczenia po korekcie masek rappingu.


In [ ]:
tmp = df_overlap.copy()

tmp["load_bin_1"] = pd.qcut(tmp["016A00219"], q=5, duplicates="drop")
tmp["load_bin_2"] = pd.qcut(tmp["016A00396"], q=5, duplicates="drop")

grouped_2d = tmp.groupby(["load_bin_1", "load_bin_2"], observed=True).agg(
    n=("U_mean", "size"),
    load_016A00219_median=("016A00219", "median"),
    load_016A00396_median=("016A00396", "median"),
    U_mean_median=("U_mean", "median"),
    U_mean_std=("U_mean", "std"),
    U_mean_p05=("U_mean", lambda x: x.quantile(0.05)),
    U_mean_p25=("U_mean", lambda x: x.quantile(0.25)),
    U_mean_p75=("U_mean", lambda x: x.quantile(0.75)),
    U_mean_p95=("U_mean", lambda x: x.quantile(0.95)),
    dust_median=(TARGET_COL, "median"),
    P_total_median=("P_total", "median"),
)

grouped_2d["U_mean_p95_minus_p05"] = (
    grouped_2d["U_mean_p95"] - grouped_2d["U_mean_p05"]
)

grouped_2d["U_mean_iqr"] = (
    grouped_2d["U_mean_p75"] - grouped_2d["U_mean_p25"]
)

grouped_2d = grouped_2d.sort_values("n", ascending=False)

grouped_2d.head(30)

In [ ]:
reliable_cells = grouped_2d[grouped_2d["n"] >= 1000].copy()

print("Number of 2D cells:", len(grouped_2d))
print("Number of reliable 2D cells:", len(reliable_cells))

reliable_cells[
    [
        "n",
        "U_mean_std",
        "U_mean_p95_minus_p05",
        "U_mean_iqr",
        "dust_median",
        "P_total_median",
    ]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])

In [ ]:
overlap_cell_summary = pd.DataFrame({
    "criterion": [
        "U_mean p95-p05 >= 1 kV",
        "U_mean p95-p05 >= 2 kV",
        "U_mean p95-p05 >= 3 kV",
        "U_mean p95-p05 >= 5 kV",
    ],
    "fraction_of_reliable_cells": [
        (reliable_cells["U_mean_p95_minus_p05"] >= 1).mean(),
        (reliable_cells["U_mean_p95_minus_p05"] >= 2).mean(),
        (reliable_cells["U_mean_p95_minus_p05"] >= 3).mean(),
        (reliable_cells["U_mean_p95_minus_p05"] >= 5).mean(),
    ],
    "fraction_of_samples_in_reliable_cells": [
        reliable_cells.loc[reliable_cells["U_mean_p95_minus_p05"] >= 1, "n"].sum() / reliable_cells["n"].sum(),
        reliable_cells.loc[reliable_cells["U_mean_p95_minus_p05"] >= 2, "n"].sum() / reliable_cells["n"].sum(),
        reliable_cells.loc[reliable_cells["U_mean_p95_minus_p05"] >= 3, "n"].sum() / reliable_cells["n"].sum(),
        reliable_cells.loc[reliable_cells["U_mean_p95_minus_p05"] >= 5, "n"].sum() / reliable_cells["n"].sum(),
    ],
})

overlap_cell_summary

In [ ]:
U_MEAN_LOW = df_overlap["U_mean"].quantile(0.01)
U_MEAN_HIGH = df_overlap["U_mean"].quantile(0.99)

df_overlap_typical_u = df_overlap[
    (df_overlap["U_mean"] >= U_MEAN_LOW)
    & (df_overlap["U_mean"] <= U_MEAN_HIGH)
].copy()

print("U_mean 1%:", U_MEAN_LOW)
print("U_mean 99%:", U_MEAN_HIGH)
print("Original:", len(df_overlap))
print("Typical U:", len(df_overlap_typical_u))
print("Removed:", len(df_overlap) - len(df_overlap_typical_u))

In [ ]:
tmp = df_overlap_typical_u.copy()

tmp["load_bin_1"] = pd.qcut(tmp["016A00219"], q=5, duplicates="drop")
tmp["load_bin_2"] = pd.qcut(tmp["016A00396"], q=5, duplicates="drop")

grouped_2d_typical_u = tmp.groupby(["load_bin_1", "load_bin_2"], observed=True).agg(
    n=("U_mean", "size"),
    U_mean_median=("U_mean", "median"),
    U_mean_std=("U_mean", "std"),
    U_mean_p05=("U_mean", lambda x: x.quantile(0.05)),
    U_mean_p95=("U_mean", lambda x: x.quantile(0.95)),
    dust_median=(TARGET_COL, "median"),
    P_total_median=("P_total", "median"),
)

grouped_2d_typical_u["U_mean_p95_minus_p05"] = (
    grouped_2d_typical_u["U_mean_p95"] - grouped_2d_typical_u["U_mean_p05"]
)

reliable_cells_typical_u = grouped_2d_typical_u[
    grouped_2d_typical_u["n"] >= 1000
].copy()

reliable_cells_typical_u[
    [
        "n",
        "U_mean_std",
        "U_mean_p95_minus_p05",
        "dust_median",
        "P_total_median",
    ]
].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])